# MolScribe matched fine-tuning - final training notebook

This notebook reproduces either project training condition using the same MolScribe commit, official starting checkpoint, clean validation set, seed, optimizer settings, and one-epoch schedule.

1. Select a **T4 GPU** runtime in Google Colab.
2. Set `CONDITION` in the first code cell to `clean` or `augmented`.
3. Upload the corresponding package when prompted.
4. Run the notebook once per condition in a fresh runtime.

The packages contain 3,758 training images and 470 clean validation images. They do not contain the synthetic test partition or the 5,088-image DECIMER benchmark. The real benchmark is reserved for final evaluation.

If the packages must be recreated, follow the data-generation and packaging commands in the project README before opening Colab.

In [ ]:
# Choose exactly one matched training condition for this Colab run.
CONDITION = 'clean'  # change to 'augmented' for the second run

PACKAGE = {
    'clean': {
        'name': 'molscribe_clean_train_validation.zip',
        'sha256': '05c42002ea761ce5ff265948cdabffef3329a5ca3b63007a758d2a4d66164252',
        'training_condition': 'clean',
        'contains_augmented_images': False,
        'drive_folder': 'molscribe_clean_probe',
    },
    'augmented': {
        'name': 'molscribe_augmented_train_clean_validation.zip',
        'sha256': '52ccd5c5eac61cd01ca55db36752da8c729ab322b628ff09b14af8bd2e90356c',
        'training_condition': 'handdrawn_augmented',
        'contains_augmented_images': True,
        'drive_folder': 'molscribe_augmented_probe',
    },
}[CONDITION]
print('Selected condition:', CONDITION)
print('Expected upload:', PACKAGE['name'])

## 1. Create the isolated MolScribe environment

The final experiment uses Python 3.10.21 and pinned legacy MolScribe dependencies without modifying Colab's base Python.

In [ ]:
import subprocess, sys
from pathlib import Path

assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'Select Runtime > Change runtime type > T4 GPU'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.10'], check=True)
subprocess.run(['uv', 'venv', '/content/molscribe_env', '--python', '3.10', '--clear'], check=True)
PY = '/content/molscribe_env/bin/python'
TORCHRUN = '/content/molscribe_env/bin/torchrun'
print('Isolated Python:', subprocess.check_output([PY, '--version'], text=True).strip())

## 2. Install pinned dependencies

In [ ]:
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--extra-index-url', 'https://download.pytorch.org/whl/cu117', 'torch==1.13.1+cu117', 'torchvision==0.14.1+cu117'], check=True)
deps = ['numpy==1.23.5', 'pandas==1.5.3', 'scipy==1.10.1', 'matplotlib==3.7.5', 'opencv-python-headless==4.5.5.64', 'Pillow==9.5.0', 'SmilesPE==0.0.3', 'rdkit==2022.9.5', 'albumentations==1.1.0', 'timm==0.4.12', 'transformers==4.30.2', 'huggingface-hub==0.16.4', 'tensorboardX==2.6.2.2', 'torchtext==0.5.0', 'configargparse==1.7', 'pyonmttok==1.37.1', 'PyYAML==6.0.1']
subprocess.run(['uv', 'pip', 'install', '--python', PY, *deps], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--no-deps', 'OpenNMT-py==2.2.0'], check=True)
subprocess.run([PY, '-c', 'import torch, torchvision, numpy, cv2, timm, onmt; print(torch.__version__, torch.cuda.is_available(), numpy.__version__)'], check=True)
assert Path(TORCHRUN).is_file(), 'torchrun was not installed in the isolated environment'

## 3. Install the exact MolScribe revision

In [ ]:
REPO = Path('/content/MolScribe')
COMMIT = '7296a30413eb55436702011efdff78131f66d162'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/thomas0809/MolScribe.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--no-deps', '-e', str(REPO)], check=True)
subprocess.run([PY, '-c', 'from molscribe.dataset import TrainDataset; from molscribe.model import Encoder, Decoder; from molscribe.loss import Criterion; print("MolScribe imports OK")'], check=True)

## 4. Upload and verify the selected synthetic package

The package hash, internal file hashes, split counts, and condition metadata are checked before extraction.

In [ ]:
from google.colab import files
import csv, hashlib, json, zipfile

uploaded = files.upload()
assert list(uploaded) == [PACKAGE['name']], f"Upload only {PACKAGE['name']}"
archive_path = Path('/content') / PACKAGE['name']
assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == PACKAGE['sha256'], 'ZIP SHA-256 mismatch'

DATA = Path('/content/aidd_molscribe_data')
DATA.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    names = set(archive.namelist())
    assert all(not name.startswith('/') and '..' not in Path(name).parts for name in names)
    meta = json.loads(archive.read('package_metadata.json'))
    assert meta['counts'] == {'train': 3758, 'validation': 470}
    assert meta['training_condition'] == PACKAGE['training_condition']
    assert meta['contains_augmented_images'] is PACKAGE['contains_augmented_images']
    assert meta['contains_real_handdrawn_images'] is False
    assert all(hashlib.sha256(archive.read(name)).hexdigest() == digest for name, digest in meta['checksums'].items())
    assert names == set(meta['checksums']) | {'package_metadata.json'}
    archive.extractall(DATA)

for split, expected in [('train', 3758), ('validation', 470)]:
    with (DATA / f'{split}.csv').open(newline='') as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == expected and all((DATA / row['file_path']).is_file() for row in rows)
print('Verified package:', CONDITION, meta['counts'])

## 5. Download and verify the official starting checkpoint

In [ ]:
CKPT = Path('/content/swin_base_char_aux_1m680k.pth')
download_code = "from huggingface_hub import hf_hub_download; import shutil; p=hf_hub_download('yujieq/MolScribe', 'swin_base_char_aux_1m680k.pth', revision='601bb0f491fb9598fa40227ae9759f800661cd74'); shutil.copyfile(p, '/content/swin_base_char_aux_1m680k.pth')"
if not CKPT.exists():
    subprocess.run([PY, '-c', download_code], check=True)
digest = hashlib.sha256()
with CKPT.open('rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        digest.update(block)
assert digest.hexdigest() == '32821fa4ba8d17f9a92e374601438c5e757947dbf1141e9ad0ef51cccfa1c501'
print('Starting checkpoint verified:', CKPT.stat().st_size, 'bytes')

## 6. Run the matched one-epoch fine-tuning condition

Both conditions use the same hyperparameters and clean validation set. The image-SMILES packages do not contain optional bond-edge annotations, so the trained output format is `chartok_coords`.

In [ ]:
OUTPUT = Path(f'/content/molscribe_{CONDITION}_probe')
OUTPUT.mkdir(exist_ok=True)
common = ['--data_path', str(DATA), '--valid_file', 'validation.csv', '--vocab_file', str(REPO / 'molscribe/vocab/vocab_chars.json'), '--formats', 'chartok_coords', '--coord_bins', '64', '--sep_xy', '--input_size', '384', '--encoder', 'swin_base', '--decoder', 'transformer', '--batch_size', '2', '--num_workers', '2', '--use_checkpoint', '--fp16', '--backend', 'nccl', '--print_freq', '50']
train_args = ['--do_train', '--train_file', 'train.csv', '--epochs', '1', '--gradient_accumulation_steps', '4', '--encoder_lr', '1e-5', '--decoder_lr', '5e-5', '--save_mode', 'best', '--seed', '20260915', '--load_path', str(CKPT), '--save_path', str(OUTPUT)]
subprocess.run([TORCHRUN, '--standalone', '--nproc_per_node=1', str(REPO / 'train.py'), *train_args, *common], cwd=REPO, check=True)
BEST = OUTPUT / 'swin_base_transformer_best.pth'
PREDICTIONS = OUTPUT / 'prediction_validation.csv'
assert BEST.is_file() and PREDICTIONS.is_file()
print('Final checkpoint:', BEST, BEST.stat().st_size, 'bytes')
print('Validation predictions:', PREDICTIONS)

## 7. Save the reproducibility record to Google Drive

This cell saves the checkpoint, validation predictions, training metadata, dependency versions, GPU name, and cryptographic hashes. Retain these files together.

In [ ]:
import shutil
from google.colab import drive

drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/AIDD_Mini_Project_3') / PACKAGE['drive_folder']
destination.mkdir(parents=True, exist_ok=True)
shutil.copy2(PREDICTIONS, destination / f'{CONDITION}_prediction_validation.csv')
if (OUTPUT / 'best_valid.json').is_file():
    shutil.copy2(OUTPUT / 'best_valid.json', destination / f'{CONDITION}_best_valid.json')
shutil.copy2(BEST, destination / BEST.name)

def file_sha256(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()

run_info = {
    'condition': CONDITION,
    'molscribe_commit': COMMIT,
    'starting_checkpoint_sha256': digest.hexdigest(),
    'final_checkpoint_sha256': file_sha256(BEST),
    'dataset_zip_sha256': PACKAGE['sha256'],
    'training_epochs': 1,
    'seed': 20260915,
    'formats': ['chartok_coords'],
    'gpu': subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip(),
    'pip_freeze': subprocess.check_output(['uv', 'pip', 'freeze', '--python', PY], text=True).splitlines(),
}
(destination / f'{CONDITION}_run_info.json').write_text(json.dumps(run_info, indent=2))
print('Saved final', CONDITION, 'results to', destination)
print('Final checkpoint SHA-256:', run_info['final_checkpoint_sha256'])